# PoetryDuel-GPT v4.1 — Lục Bát Quality Reinforcement

**~540K window-1 Lục Bát đối thơ pairs, 5 rules: Rhyme + Tone + Syllable + Trầm-Bổng + Nhịp điệu.**

| Step | Time |
|---|---|
| Download data + train tokenizer | ~2 min |
| Training (10K steps) | ~2 hr T4 |
| Generate + evaluate poetry | ~2 min |

### v4.1 Changes from v4:
- ❌ **Thất Ngôn removed** (moved to v5) — no more TN data dilution
- ❌ **T2a TN re-split removed** — no more post-processing hacks
- ❌ **T2b weighted loss removed** — simple cross-entropy, pure Lục Bát
- ✅ **R4 Trầm-Bổng added** — `[TRAMBONG:NH]`/`[TRAMBONG:HN]` control token
- ✅ **5-rule evaluation** — Rhyme + Tone + Syllable + Trầm-Bổng + Nhịp điệu
- ✅ **P3 syllable enforcement = optional** (off for eval, on for UI)

### v4.1 Targets
| Metric | Target |
|---|---|
| R1 Rhyme (pos6) | 65%+ |
| R2 Tone (BTB+BTBB) | 93%+ |
| R3 Syllable (6+8) | 85%+ |
| R4 Trầm-Bổng | 60%+ |
| R5 Nhịp điệu | 75%+ |
| All 5 pass | 30%+ |

In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm datasets
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print('\nRun cells in order.')


In [ ]:
# @title 2. Download Data + Train Tokenizer (~2 min)
from google.colab import drive
drive.mount('/content/drive')

# Copy data.zip from Drive
!cp /content/drive/MyDrive/poetry-dual-gpt/data.zip /content/poetry-dual-gpt/ 2>/dev/null
!unzip -o data.zip

!mkdir -p data
!mv doi_tho_corpus.txt data/doi_tho_corpus.txt 2>/dev/null
!ls data/

# Train BPE tokenizer on v4.1 corpus (includes [TRAMBONG:NH]/[TRAMBONG:HN])
%cd /content/poetry-dual-gpt
!python src/train_bpe.py --corpus data/doi_tho_corpus.txt

# Verify tokenizer
from tokenizers import Tokenizer
tok = Tokenizer.from_file('tokenizer/poetry_bpe.model')
print(f'Vocab: {tok.get_vocab_size():,}')

key_tokens = [
    '<|pad|>', '<|start|>', '<|reply|>', '<|end|>',
    '[LUC_BAT]', '<|linebreak|>',
    '[RHYME:ong]', '[TONE:BBTBBT]',
    '[TRAMBONG:NH]', '[TRAMBONG:HN]'
]
for name in key_tokens:
    tid = tok.token_to_id(name)
    n = len(tok.encode(name).ids)
    status = 'OK' if n == 1 else 'FAIL'
    print(f'  {name:25s} id={tid:5d} subwords={n} {status}')

# Verify v4.1: no Thất Ngôn, only Lục Bát
import re
with open('data/doi_tho_corpus.txt', encoding='utf-8') as f:
    corpus = f.read()
lb = len(re.findall(r'\[LUC_BAT\]', corpus))
tn = len(re.findall(r'\[THAT_NGON\]', corpus))
tb_nh = len(re.findall(r'\[TRAMBONG:NH\]', corpus))
tb_hn = len(re.findall(r'\[TRAMBONG:HN\]', corpus))
print(f'\nLục Bát pairs: {lb:,}')
print(f'Thất Ngôn pairs: {tn:,} (should be 0)')
print(f'Trầm-Bổng NH: {tb_nh:,} | HN: {tb_hn:,}')

assert tn == 0, 'ERROR: Thất Ngôn found in v4.1 corpus!'
assert lb > 0, 'ERROR: No Lục Bát pairs!'
print('\n✓ v4.1 corpus verified: Lục Bát only, Trầm-Bổng tokens present')
print('\nReady for training.')


In [ ]:
# @title 3. Train — 10K steps, pure Lục Bát (~2 hr T4)
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU"

%cd /content/poetry-dual-gpt
import glob

# v4.1: Pure Lục Bát, no weighted loss, simple CE
CORPUS = 'data/doi_tho_corpus.txt'

latest = sorted(glob.glob("checkpoints/doi_tho_step_*.pt"))
if latest:
    print(f"Resuming from {latest[-1]}")
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS} --resume {latest[-1]}
else:
    print("Training from scratch on v4.1 LB-only corpus")
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS}

print("\nTraining complete -> checkpoints/doi_tho_best.pt")


In [ ]:
# @title 4. Evaluate: Run 5-Rule Benchmark
%cd /content/poetry-dual-gpt

!PYTHONPATH=. python evaluate/eval_rules.py

print('\nSee documents/rule_evaluation.md for full report')


In [ ]:
# @title 5. Test Generation (Lục Bát)
%cd /content/poetry-dual-gpt
import os, glob

ckpt = 'checkpoints/doi_tho_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/doi_tho_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 1000000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('No checkpoint found.')
else:
    print(f'Testing: {ckpt}\n')
    
    print('--- Lục Bát: Single line → couplet ---')
    !python src/sample.py --checkpoint {ckpt} \
        --prompt "Than em nhu chen lua dong" \
        --num_samples 3 --device cuda
    
    print('\n--- Lục Bát: Couplet → couplet (doi tho) ---')
    !python src/sample.py --checkpoint {ckpt} \
        --prompt "Than em nhu chen lua dong | Phat pho duoi ngon nang hong ban mai" \
        --num_samples 3 --device cuda


In [ ]:
# @title 6. Generate Custom Poem
%cd /content/poetry-dual-gpt

# Single 6-syl prompt (auto-wraps with [LUC_BAT] [RHYME:X] [TONE:BBBBBB] [TRAMBONG:NH])
!python src/sample.py \
    --checkpoint checkpoints/doi_tho_best.pt \
    --prompt "Mua xuan lich thich vuon dao" \
    --temperature 0.75 --top_p 0.92 --num_samples 2 --device cuda

# Multi-couplet input (use | between lines)
!python src/sample.py \
    --checkpoint checkpoints/doi_tho_best.pt \
    --prompt "Kieu nhi phan mong nhu to | Mot loi da loi toc to voi chang" \
    --temperature 0.75 --top_p 0.92 --num_samples 2 --device cuda


In [ ]:
# @title 7. Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/poetry-dual-gpt'
!mkdir -p {DRIVE_DIR}/checkpoints

!cp -v tokenizer/poetry_bpe.model {DRIVE_DIR}/
import glob
for ckpt in sorted(glob.glob('checkpoints/doi_tho_*.pt')):
    !cp -v {ckpt} {DRIVE_DIR}/checkpoints/
print('\nSaved to Drive.')
